In [ ]:
import os
work_dir = "H:\workspace\\bott"
os.chdir(work_dir)
print("Current working dir: ", os.getcwd())

In [ ]:
from time import time
import numpy as np
import cupy as cp

In [ ]:
from bott.physics_models import simulate_potential
from bott.utils import print_system_info

In [ ]:
print_system_info()

In [ ]:
params_abTEM = {  
    # Device configuration
    "device_abtem": "cpu",

    # Crystal structure input
    "path_crystal": "./data/SrTiO3.cif",

    # Potential parameters
    "potential_extent_x": 62.6,  # Angstrom
    "potential_extent_y": 62.6,  # Angstrom
    "lateral_sampling": 0.2 * 2 / 3,  # Angstrom
    "vertical_sampling": 2,
    "potential_parametrization": "lobato",  # or "kirkland"
    "potential_projection": "finite",       # or "infinite"

    # Phonon parameters
    "random_seed": 42,
    "use_frozen_phonon": False,
    "num_phonon_configs": 5,
    "phonon_sigma": {
        'Sr': 0.088,
        'Ti': 0.0746,
        'O': 0.0963,
    },

    # Probe parameters
    "energy": 200e3,  # in eV
    "convergence_angle": 19.1,  # mrad
    "df": 0,
    "aberrations": {},

    # Scan parameters
    "scan_step_size": 0.3,  # nm
    "return_pacbed": True,
}

In [ ]:
# Benchmark conditions
thickness_series = np.arange(10, 60, 10)
repeat = 10
thickness_series

In [ ]:
# CPU timing

compute_mode = 'cpu'
params_abTEM['device_abtem'] = compute_mode
print(f"Currently benchmarking with '{compute_mode}'")

thickness_times_cpu = []

for thickness in thickness_series:
    print(f"Thickness: {thickness} Ang")
    time_report = 0
    for n in range(repeat):
        
        time_start = time()
        potential = simulate_potential(thickness, params_abTEM)
        time_end = time()
        
        run_time = time_end - time_start
        print(f"Run time = {run_time:.3f} sec")
        time_avg = run_time / repeat
    time_avg += time_report
    print(f"Averaged time = {run_time:.3f} sec")
    
    thickness_times_cpu.append(time_avg)

In [ ]:
# GPU timing

import gc # garbage collection

compute_mode = 'gpu'
params_abTEM['device_abtem'] = compute_mode
print(f"Currently benchmarking with '{compute_mode}'")

thickness_times_gpu = []

# Create CUDA events
start_event = cp.cuda.Event()
end_event = cp.cuda.Event()

for thickness in thickness_series:
    print(f"Thickness: {thickness} Ang")
    time_report = 0
    for n in range(repeat):
        
        
        start_event.record()
        potential = simulate_potential(thickness, params_abTEM)
        end_event.record()
        
        end_event.synchronize()
        run_time = cp.cuda.get_elapsed_time(start_event, end_event) / 1000 # cp event is in unit of ms
        print(f"Run time = {run_time:.3f} sec")
        time_avg = run_time / repeat
        
        # Cleanup
        del potential
        cp._default_memory_pool.free_all_blocks()
        gc.collect()
        
    time_avg += time_report
    print(f"Averaged time = {run_time:.3f} sec")
    
    thickness_times_gpu.append(time_avg)


In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.title('Potential generation time')
plt.scatter(thickness_series, thickness_times_cpu, label='cpu')
plt.scatter(thickness_series, thickness_times_gpu, label='gpu')
plt.legend()
plt.show()